# Generazione PNG per reti violenza

Questo notebook legge i CSV già esportati dalla pipeline reti e genera i PNG in una cartella `png/`.

In [20]:
from pathlib import Path
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
INPUT_DIR = Path(r'output\reports\step_3.2')
OUTPUT_DIR = Path(r'output\reports\step_3.3')
PNG_DIR = OUTPUT_DIR / 'png'
PNG_DIR.mkdir(parents=True, exist_ok=True)

PROSPETTO_RETI = INPUT_DIR / 'prospetto_reti_per_regione.csv'
PROSPETTO_AMBITO = INPUT_DIR / 'prospetto_ambito_per_regione.csv'
PROSPETTO_TIPO10 = INPUT_DIR / 'prospetto_soggetti_tipo10_per_regione.csv'
PROSPETTO_TIPO30 = INPUT_DIR / 'prospetto_soggetti_tipo30_per_regione.csv'
TABELLA_SOGGETTI = INPUT_DIR / 'tabella_soggetti.csv'
TABELLA_RETI = INPUT_DIR / 'tabella_reti.csv'

for p in [PROSPETTO_RETI, PROSPETTO_AMBITO, PROSPETTO_TIPO10, PROSPETTO_TIPO30, TABELLA_SOGGETTI, TABELLA_RETI]:
    print(p, 'OK' if p.exists() else 'MISSING')


output\reports\step_3.2\prospetto_reti_per_regione.csv OK
output\reports\step_3.2\prospetto_ambito_per_regione.csv OK
output\reports\step_3.2\prospetto_soggetti_tipo10_per_regione.csv OK
output\reports\step_3.2\prospetto_soggetti_tipo30_per_regione.csv OK
output\reports\step_3.2\tabella_soggetti.csv OK
output\reports\step_3.2\tabella_reti.csv OK


In [21]:
def read_csv_flexible(path: Path) -> pd.DataFrame:
    encodings = ['utf-8', 'utf-8-sig', 'latin1', 'cp1252']
    seps = [',', ';', '\t']
    last_error = None
    for enc in encodings:
        for sep in seps:
            try:
                return pd.read_csv(path, encoding=enc, sep=sep)
            except Exception as e:
                last_error = e
    raise RuntimeError(f'Impossibile leggere {path}: {last_error}')

def autosize_height(n_rows: int, base: float = 4.0, scale: float = 0.20, max_size: float = 16.0) -> float:
    return min(max_size, max(base, base + n_rows * scale))

def save_barh(df: pd.DataFrame, y_col: str, x_col: str, title: str, out_path: Path) -> None:
    if df.empty:
        print('skip', out_path.name, '(empty)')
        return
    plot_df = df.copy().sort_values(x_col, ascending=True)
    h = autosize_height(len(plot_df), base=4.0, scale=0.20, max_size=16.0)
    plt.figure(figsize=(12, h))
    plt.barh(plot_df[y_col].astype(str), plot_df[x_col])
    plt.title(title)
    plt.xlabel('Numero')
    plt.ylabel('')
    plt.tight_layout()
    plt.savefig(out_path, dpi=200, bbox_inches='tight')
    plt.close()
    print('saved', out_path)

def save_figura4_grouped(df: pd.DataFrame, out_path: Path) -> None:
    if df.empty:
        print('skip', out_path.name, '(empty)')
        return
    plot_df = df.copy()
    x = range(len(plot_df))
    width = 0.42
    plt.figure(figsize=(12, autosize_height(len(plot_df), base=4.0, scale=0.15, max_size=10.0)))
    plt.barh([i - width / 2 for i in x], plot_df['soggetti_proponenti'], height=width, label='Soggetti proponenti')
    plt.barh([i + width / 2 for i in x], plot_df['attori_coinvolti'], height=width, label='Attori coinvolti')
    plt.yticks(list(x), plot_df['regione'].astype(str))
    plt.title('Figura 4. Soggetti proponenti e attori coinvolti per regione')
    plt.xlabel('Numero')
    plt.ylabel('')
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=200, bbox_inches='tight')
    plt.close()
    print('saved', out_path)


In [22]:
tab_soggetti = read_csv_flexible(TABELLA_SOGGETTI)
tab_reti = read_csv_flexible(TABELLA_RETI)
p_reti = read_csv_flexible(PROSPETTO_RETI)
p_ambito = read_csv_flexible(PROSPETTO_AMBITO)
p_tipo10 = read_csv_flexible(PROSPETTO_TIPO10)
p_tipo30 = read_csv_flexible(PROSPETTO_TIPO30)

tab_soggetti.head(), tab_reti.head()

(                                             id_rete  \
 0  13_2406_rta_13_241122103901_4166---Protocollir...   
 1  18_2406_rta_18_240926103211_4166---PROTOCOLLOv...   
 2  16_2406_rta_16_250410170705_4166---protocolloP...   
 3  12_2406_rta_12_241004144338_4166---06-Protocol...   
 4  18_2406_rta_18_240926102706_4166---protocollom...   
 
                         input_json  \
 0  all_risultati_enriched_2.4.json   
 1  all_risultati_enriched_2.4.json   
 2  all_risultati_enriched_2.4.json   
 3  all_risultati_enriched_2.4.json   
 4  all_risultati_enriched_2.4.json   
 
                                          titolo_rete  \
 0  2406_rta_13_241122103901_4166---Protocollirete...   
 1  2406_rta_18_240926103211_4166---PROTOCOLLOviol...   
 2  2406_rta_16_250410170705_4166---protocolloPref...   
 3  2406_rta_12_241004144338_4166---06-ProtocolloC...   
 4  2406_rta_18_240926102706_4166---protocollomond...   
 
                                                 file   regione  provincia  

In [23]:
# Figura 1: soggetti proponenti per tipo 10
if {'ruolo_proponente', 'tipo_aggregato_10', 'nome_soggetto'}.issubset(tab_soggetti.columns):
    figura1 = (tab_soggetti.loc[tab_soggetti['ruolo_proponente'] == 1]
               .groupby('tipo_aggregato_10', dropna=False)
               .agg(numero=('nome_soggetto', 'size'))
               .reset_index()
               .rename(columns={'tipo_aggregato_10': 'tipologia'})
               .sort_values(['numero', 'tipologia'], ascending=[False, True]))
else:
    figura1 = pd.DataFrame(columns=['tipologia', 'numero'])

save_barh(figura1, 'tipologia', 'numero', 'Figura 1. Soggetti proponenti per aggregazione 10', PNG_DIR / 'figura1_soggetti_promotori_per_tipo10.png')

saved output\reports\step_3.3\png\figura1_soggetti_promotori_per_tipo10.png


In [24]:
# Figura 2: soggetti proponenti per regione
if {'ruolo_proponente', 'regione', 'nome_soggetto'}.issubset(tab_soggetti.columns):
    figura2 = (tab_soggetti.loc[tab_soggetti['ruolo_proponente'] == 1]
               .groupby('regione', dropna=False)
               .agg(numero=('nome_soggetto', 'size'))
               .reset_index())
else:
    figura2 = pd.DataFrame(columns=['regione', 'numero'])

save_barh(figura2, 'regione', 'numero', 'Figura 2. Soggetti proponenti per regione', PNG_DIR / 'figura2_soggetti_proponenti_per_regione.png')

saved output\reports\step_3.3\png\figura2_soggetti_proponenti_per_regione.png


In [25]:
# Figura 3: attori coinvolti per tipo 10
if {'ruolo_attore', 'tipo_aggregato_10', 'nome_soggetto'}.issubset(tab_soggetti.columns):
    figura3 = (tab_soggetti.loc[tab_soggetti['ruolo_attore'] == 1]
               .groupby('tipo_aggregato_10', dropna=False)
               .agg(numero=('nome_soggetto', 'size'))
               .reset_index()
               .rename(columns={'tipo_aggregato_10': 'tipologia'})
               .sort_values(['numero', 'tipologia'], ascending=[False, True]))
else:
    figura3 = pd.DataFrame(columns=['tipologia', 'numero'])

save_barh(figura3, 'tipologia', 'numero', 'Figura 3. Attori coinvolti per aggregazione 10', PNG_DIR / 'figura3_attori_coinvolti_per_tipo10.png')

saved output\reports\step_3.3\png\figura3_attori_coinvolti_per_tipo10.png


In [26]:
# Figura 4: proponenti vs attori per regione
if {'ruolo_proponente', 'ruolo_attore', 'regione', 'nome_soggetto'}.issubset(tab_soggetti.columns):
    prop = (tab_soggetti.loc[tab_soggetti['ruolo_proponente'] == 1]
            .groupby('regione', dropna=False)
            .agg(soggetti_proponenti=('nome_soggetto', 'size'))
            .reset_index())
    att = (tab_soggetti.loc[tab_soggetti['ruolo_attore'] == 1]
           .groupby('regione', dropna=False)
           .agg(attori_coinvolti=('nome_soggetto', 'size'))
           .reset_index())
    figura4 = prop.merge(att, on='regione', how='outer').fillna(0)
    for c in ['soggetti_proponenti', 'attori_coinvolti']:
        figura4[c] = figura4[c].astype(int)
else:
    figura4 = pd.DataFrame(columns=['regione', 'soggetti_proponenti', 'attori_coinvolti'])

save_figura4_grouped(figura4, PNG_DIR / 'figura4_proponenti_vs_attori_per_regione.png')

saved output\reports\step_3.3\png\figura4_proponenti_vs_attori_per_regione.png


In [27]:
# Figura 5: ambiti territoriali
if not tab_reti.empty and {'ambito_territoriale', 'id_rete'}.issubset(tab_reti.columns):
    figura5 = (tab_reti.groupby('ambito_territoriale', dropna=False)
               .agg(numero=('id_rete', 'nunique'))
               .reset_index()
               .rename(columns={'ambito_territoriale': 'ambito'})
               .sort_values(['numero', 'ambito'], ascending=[False, True]))
else:
    figura5 = pd.DataFrame(columns=['ambito', 'numero'])

save_barh(figura5, 'ambito', 'numero', 'Figura 5. Ambiti territoriali delle reti', PNG_DIR / 'figura5_ambiti_territoriali.png')

saved output\reports\step_3.3\png\figura5_ambiti_territoriali.png


In [28]:
readme = [
    'PNG generati:',
    '- figura1_soggetti_promotori_per_tipo10.png',
    '- figura2_soggetti_proponenti_per_regione.png',
    '- figura3_attori_coinvolti_per_tipo10.png',
    '- figura4_proponenti_vs_attori_per_regione.png',
    '- figura5_ambiti_territoriali.png',
]
(PNG_DIR / 'README_png.txt').write_text('\n'.join(readme), encoding='utf-8')
print('Creato:', PNG_DIR / 'README_png.txt')

Creato: output\reports\step_3.3\png\README_png.txt
